# GreenGrid AI — Weather + éCO2mix exploration (Aziz, Phase 0, audited v3)

**Run on Kaggle:** Enable Internet, upload this `.ipynb`, and Run All. This version fixes the source-cadence audit and generates a shareable technical summary. All empirical conclusions require the cells to execute on genuine data.

This **audited v3** was executed in Kaggle on 1 November 2026. Its preserved outputs document the quality audit, correlations, and archived weather forecast pilot. Large inline images are stored separately under `reports/figures/aziz_phase0/`.

The year **2025 is never fetched** and remains a locked test in the proposed study protocol.

**Methodology warning:** ERA5 reanalysis at target time is not known in advance. Use it for exploratory analysis or an explicitly named perfect-weather bound—not as an operational day-ahead input. `_previous_day1` is a fixed 24-hour lead forecast, not necessarily a forecast snapshot from one common issuance time. RTE `taux_co2` represents generation-based average emissions, not marginal avoided emissions.


In [21]:
import io, time
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

OUT = Path('/kaggle/working/greengrid_phase0') if Path('/kaggle/working').exists() else Path('greengrid_phase0')
OUT.mkdir(parents=True, exist_ok=True)
START, END = '2024-01-01', '2024-12-31'
CITIES = {
 'Paris':(48.8566,2.3522), 'Lyon':(45.7640,4.8357),
 'Marseille':(43.2965,5.3698), 'Lille':(50.6292,3.0573),
 'Bordeaux':(44.8378,-0.5792), 'Toulouse':(43.6047,1.4442),
 'Nantes':(47.2184,-1.5536), 'Strasbourg':(48.5734,7.7521)
}
retry = Retry(total=4, connect=4, read=4, backoff_factor=2, status_forcelist=[429,500,502,503,504])
S = requests.Session()
S.mount('https://', HTTPAdapter(max_retries=retry))
print('Output:', OUT, '| cities:', len(CITIES))

Output: /kaggle/working/greengrid_phase0 | cities: 8


## 1. Observed historical weather — ERA5

For **EDA only**, download hourly temperature at 2 m, wind speed at 100 m, and solar radiation for eight French city coordinates. These are indicative observation locations, **not** weighted weather samples representative of France. For future modeling, population/park-capacity weights need independent sourced metadata.

In [22]:
def get_weather_city(name,lat,lon):
    endpoint = 'https://archive-api.open-meteo.com/v1/archive'
    params = dict(latitude=lat,longitude=lon,start_date=START,end_date=END,
                  hourly='temperature_2m,wind_speed_100m,shortwave_radiation',
                  timezone='UTC',models='era5')
    res=S.get(endpoint,params=params,timeout=120)
    res.raise_for_status(); payload=res.json()
    if payload.get('error') or not payload.get('hourly'):
        raise RuntimeError(f'Open-Meteo returned error for {name}: {payload}')
    hourly=payload['hourly']
    df=pd.DataFrame(hourly)
    df['time_utc']=pd.to_datetime(df.pop('time'),utc=True,errors='raise')
    df['city']=name
    return df[['time_utc','city','temperature_2m','wind_speed_100m','shortwave_radiation']]

try:
    wx_raw=pd.concat([get_weather_city(n,*coords) for n,coords in CITIES.items()],ignore_index=True)
    wx_raw.to_csv(OUT/'weather_2024_8_cities.csv',index=False)
    print('Weather downloaded:',wx_raw.shape,'| 2025 not accessed')
except Exception as e:
    wx_raw=None
    print('WEATHER UNAVAILABLE; check Kaggle Internet/API:', type(e).__name__, str(e)[:350])
    print('No synthetic weather substituted.')

Weather downloaded: (70272, 5) | 2025 not accessed


## 2. RTE éCO2mix dataset — directly from ODRÉ

Use **consolidated/final** `eco2mix-national-cons-def`. Select `date_heure`, `consommation`, `taux_co2` only. RTE documents actual consumption/production primarily at half-hourly resolution, even though this 2024 export may contain quarter-hour timestamps. Do not infer independent measurement frequency from row counts alone; inspect duplicate values (section 6). Aggregate the timestamped values to UTC hourly means for the two plots. A GET to the Opendatasoft export endpoint is attempted. If it fails, use the manual-CSV fallback below from the official export page.

In [23]:
RTE_EXPORT='https://odre.opendatasoft.com/api/explore/v2.1/catalog/datasets/eco2mix-national-cons-def/exports/csv'
RTE_PAGE='https://odre.opendatasoft.com/explore/dataset/eco2mix-national-cons-def/export/'

def parse_rte(raw):
    frame=pd.read_csv(io.BytesIO(raw),sep=None,engine='python',dtype=str,encoding='utf-8-sig')
    frame.columns=[x.strip().replace('\ufeff','') for x in frame.columns]
    needed=['date_heure','consommation','taux_co2']
    if not set(needed).issubset(frame.columns):
        raise ValueError(f'Unexpected CSV fields: {frame.columns.tolist()[:15]}')
    frame=frame[needed].copy()
    frame['time_utc']=pd.to_datetime(frame['date_heure'],utc=True,errors='coerce')
    for v in ['consommation','taux_co2']:
        frame[v]=pd.to_numeric(frame[v].str.replace(',','.',regex=False),errors='coerce')
    frame=frame.dropna(subset=['time_utc']).sort_values('time_utc')
    frame=frame[(frame.time_utc>=pd.Timestamp('2024-01-01',tz='UTC')) &
                (frame.time_utc<pd.Timestamp('2025-01-01',tz='UTC'))]
    if frame.empty: raise ValueError('No 2024 rows in CSV')
    return frame

try:
    query={
       'select':'date_heure,consommation,taux_co2',
       'where':"date_heure >= '2024-01-01T00:00:00Z' AND date_heure < '2025-01-01T00:00:00Z'",
       'delimiter':';'
    }
    resp=S.get(RTE_EXPORT,params=query,timeout=180)
    resp.raise_for_status()
    rte_half=parse_rte(resp.content)
    print('RTE rows:',len(rte_half))
except Exception as e:
    rte_half=None
    print('RTE export could not be downloaded automatically:',type(e).__name__,str(e)[:450])
    print('Fallback: download only year 2024 as CSV from',RTE_PAGE)
    print('Then upload it as a Kaggle dataset and set MANUAL_RTE_FILE in the next cell.')

RTE rows: 35136


In [24]:
MANUAL_RTE_FILE = None  # e.g. '/kaggle/input/my-rte-data/eco2mix_2024.csv'
if rte_half is None and MANUAL_RTE_FILE:
    rte_half=parse_rte(Path(MANUAL_RTE_FILE).read_bytes())
    print('Manual RTE import succeeded:',len(rte_half))
if rte_half is not None:
    rte_hourly=(rte_half.set_index('time_utc')[['consommation','taux_co2']]
                .resample('1h').mean().rename(columns={'consommation':'consumption_MW',
                                                       'taux_co2':'co2_production_g_per_kWh'}))
    rte_hourly.to_csv(OUT/'rte_2024_hourly.csv')
    print('Hourly RTE rows:',len(rte_hourly))
else:
    rte_hourly=None
    print('RTE data not available; no made-up plot will be shown.')

Hourly RTE rows: 8784


## 3. Join and two required genuine figures

Simple unweighted city means are used **for exploratory graphics only**. They do not represent national temperature or wind farm production. The timestamp is UTC throughout, avoiding ambiguous autumn clock-hour labels. Each figure uses the actual joined observations and is generated only if both datasets are present.

In [25]:
if wx_raw is not None and rte_hourly is not None:
    wx_mean=(wx_raw.groupby('time_utc')[['temperature_2m','wind_speed_100m','shortwave_radiation']]
                        .mean().rename(columns={'temperature_2m':'mean_city_temp_C',
                                                'wind_speed_100m':'mean_city_wind_100m_kmh',
                                                'shortwave_radiation':'mean_city_solar_W_m2'}))
    joined=rte_hourly.join(wx_mean,how='inner')
    joined.to_csv(OUT/'joined_2024_hourly.csv')
    assert len(joined)>8000, f'Unexpectedly few joined timestamps: {len(joined)}'
    quality=pd.DataFrame({'missing':joined.isna().sum(), 'non_null':joined.notna().sum(),
                          'missing_fraction':joined.isna().mean()})
    quality.to_csv(OUT/'data_quality_2024.csv')
    print('Joined hours:',len(joined), '\nMissing percentages:\n',quality['missing_fraction'].mul(100).round(2))
    # hourly pairs can be highly autocorrelated: correlations are descriptive, not causal.
    pairs=[('mean_city_temp_C','consumption_MW','Temperature (mean across 8 cities, °C)',
            'Consumption (MW)','fig_temp_vs_consumption_2024.png'),
           ('mean_city_wind_100m_kmh','co2_production_g_per_kWh',
            'Wind at 100 m (mean across 8 cities, km/h)',
            'RTE production CO2 intensity (g/kWh)','fig_wind_vs_carbon_2024.png')]
    for x,y,xlab,ylab,fname in pairs:
        subset=joined[[x,y]].dropna()
        fig,ax=plt.subplots(figsize=(8,5))
        ax.scatter(subset[x],subset[y],s=4,alpha=.22,rasterized=True)
        ax.set_xlabel(xlab);ax.set_ylabel(ylab)
        ax.set_title(f'France 2024 — {fname.replace("fig_","").replace("_2024.png","").replace("_"," ")}')
        ax.grid(alpha=.15)
        fig.tight_layout();fig.savefig(OUT/fname,dpi=170);plt.show();plt.close(fig)
        print(fname, '| n=',len(subset),'| Pearson r=',round(subset[x].corr(subset[y]),3))
else:
    print('Figure generation skipped: missing genuine input feed(s). No fabricated results.')

Joined hours: 8784 
Missing percentages:
 consumption_MW              0.01
co2_production_g_per_kWh    0.01
mean_city_temp_C            0.00
mean_city_wind_100m_kmh     0.00
mean_city_solar_W_m2        0.00
Name: missing_fraction, dtype: float64


<Figure size 800x500 with 1 Axes>

fig_temp_vs_consumption_2024.png | n= 8783 | Pearson r= -0.575


<Figure size 800x500 with 1 Axes>

fig_wind_vs_carbon_2024.png | n= 8783 | Pearson r= -0.007


## 4. Optional fixed-lead forecast archive pilot (not ERA5)

`_previous_day1` means the value predicted **24 h before the target hour**. This is suitable for a fixed-lead demonstration, not a perfect reconstruction of a daily decision at a fixed issuance time. Most archives begin in 2024; for an operational J+1 study requiring a fixed issued run, inspect the Single Runs API and model-specific archive availability.

In [26]:
PREVIOUS_RUNS='https://previous-runs-api.open-meteo.com/v1/forecast'
try:
    params=dict(latitude=48.8566, longitude=2.3522, start_date='2024-10-01',
                end_date='2024-10-07', timezone='UTC',
                hourly='temperature_2m_previous_day1,wind_speed_100m_previous_day1')
    rr=S.get(PREVIOUS_RUNS,params=params,timeout=120)
    rr.raise_for_status()
    pilot=rr.json()
    if pilot.get('error'): raise RuntimeError(str(pilot))
    if 'hourly' not in pilot: raise RuntimeError('No hourly data')
    piloth=pd.DataFrame(pilot['hourly'])
    piloth.to_csv(OUT/'openmeteo_previous_day1_PILOT_oct2024.csv',index=False)
    print('Historical fixed-lead pilot downloaded:',piloth.shape)
    print('Missing fraction by variable:',piloth.isna().mean().round(3).to_dict())
except Exception as e:
    print('Archive pilot not confirmed in this runtime:',type(e).__name__,str(e)[:450])
    print('Check exact model coverage before citing it as available for the full project.')

Historical fixed-lead pilot downloaded: (168, 3)
Missing fraction by variable: {'time': 0.0, 'temperature_2m_previous_day1': 0.0, 'wind_speed_100m_previous_day1': 0.0}


## 5. Defensible conclusion for Evaluation 1

- Phase 0 requires **feasibility, not a trained model**.
- Document the **source of the target** and the difference between RTE's *production-based* `taux_co2` and imported/consumption/marginal emissions.
- Never use future observed ERA5 weather as though it were known when the forecast was issued.
- Proposed locked split: train 2019–2023, validate 2024, test 2025 **for baselines and no-future-weather features**. A separate operational-forecast track requires compatible forecast archives and a **revised, agreed split**.
- Coordinate target schema, emissions meaning, and forecast-issue time with Ramy and supervisors.

## 6. RTE cadence, missingness and UTC timestamp audit — corrected

The official **consolidated / final** dataset reports actual consumption and production on a **30-minute** cadence, while some **forecasts** have a 15-minute cadence. Therefore 15-minute export timestamps with null values for consumption or CO₂ can be expected. The original v2 'same as previous' comparison was not useful for inferring cadence because intervening rows were null; this version displays **non-null coverage by minute** instead.

We independently audit repeated UTC timestamps, unexpected gaps in the exported quarter-hour timestamp grid, and hours with a **non-null target**. Do not equate 'all hours exist in the index' with 'all hours contain valid target values'. We do **not** infer DST is the cause of a detected UTC anomaly without inspecting the raw records.


In [27]:
if rte_half is not None:
    check = rte_half[['time_utc','consommation','taux_co2']].copy().sort_values('time_utc')
    check['minute_utc'] = check['time_utc'].dt.minute
    nonnull_by_minute = (check.groupby('minute_utc')
                         .agg(rows=('time_utc','size'),
                              consumption_present=('consommation',lambda z: int(z.notna().sum())),
                              co2_present=('taux_co2',lambda z: int(z.notna().sum()))))
    for col in ['consumption_present','co2_present']:
        nonnull_by_minute[col+'_pct'] = (100 * nonnull_by_minute[col] / nonnull_by_minute['rows']).round(2)
    nonnull_by_minute.to_csv(OUT/'rte_nonnull_coverage_by_timestamp_minute.csv')

    duplicated = check.loc[check['time_utc'].duplicated(keep=False)]
    duplicate_utc_count = int(check['time_utc'].duplicated().sum())
    distinct_utc_timestamps = pd.DatetimeIndex(check['time_utc'].drop_duplicates())
    expected_qh = pd.date_range('2024-01-01','2025-01-01',freq='15min',tz='UTC',inclusive='left')
    missing_qh = expected_qh.difference(distinct_utc_timestamps)
    unexpected_utc = distinct_utc_timestamps.difference(expected_qh)
    full_hours = pd.date_range('2024-01-01','2025-01-01',freq='h',tz='UTC',inclusive='left')
    absent_hours = full_hours.difference(rte_hourly.index)
    empty_consumption_hours = rte_hourly.index[rte_hourly['consumption_MW'].isna()]
    empty_carbon_hours = rte_hourly.index[rte_hourly['co2_production_g_per_kWh'].isna()]

    qa = pd.DataFrame([
        ['Export rows',len(check)],
        ['Unique UTC timestamps',len(distinct_utc_timestamps)],
        ['Duplicate UTC timestamps (extra rows)',duplicate_utc_count],
        ['Expected UTC 15-minute slots',len(expected_qh)],
        ['Missing UTC 15-minute slots',len(missing_qh)],
        ['Unexpected UTC timestamps off expected grid',len(unexpected_utc)],
        ['Null consumption export rows',int(check['consommation'].isna().sum())],
        ['Null CO2 export rows',int(check['taux_co2'].isna().sum())],
        ['Hours missing from resampled index',len(absent_hours)],
        ['Hours with no valid consumption',len(empty_consumption_hours)],
        ['Hours with no valid CO2 intensity',len(empty_carbon_hours)],
    ],columns=['check','value'])
    qa.to_csv(OUT/'rte_timestamp_quality_checks_corrected.csv',index=False)
    print('Observed non-null measurement coverage by UTC timestamp minute:')
    display(nonnull_by_minute)
    print('Quality audit:')
    display(qa)
    if duplicate_utc_count:
        print('Duplicated timestamps — source rows (inspect before training):')
        display(duplicated.head(12))
    if len(missing_qh):
        print('Missing 15-minute UTC timestamp slots (up to 12):',list(missing_qh[:12]))
    if len(empty_consumption_hours) or len(empty_carbon_hours):
        print('Hours lacking valid targets (first 12) — consumption:',list(empty_consumption_hours[:12]),
              '; carbon:',list(empty_carbon_hours[:12]))
    print('Note: source-null 15-minute rows can be expected when realized data is half-hourly.')
    if duplicate_utc_count or len(empty_consumption_hours) or len(empty_carbon_hours):
        print('ACTION: Resolve anomalies or document the exclusion policy before model training.')
else:
    print('RTE sampling audit skipped; load real RTE data first.')


Observed non-null measurement coverage by UTC timestamp minute:


            rows  consumption_present  co2_present  consumption_present_pct  \
minute_utc                                                                    
0           8784                 8784         8784                    100.0   
15          8784                    0            0                      0.0   
30          8784                 8784         8784                    100.0   
45          8784                    0            0                      0.0   

            co2_present_pct  
minute_utc                   
0                     100.0  
15                      0.0  
30                    100.0  
45                      0.0  

Quality audit:


                                          check  value
0                                   Export rows  35136
1                         Unique UTC timestamps  35132
2         Duplicate UTC timestamps (extra rows)      4
3                  Expected UTC 15-minute slots  35136
4                   Missing UTC 15-minute slots      4
5   Unexpected UTC timestamps off expected grid      0
6                  Null consumption export rows  17568
7                          Null CO2 export rows  17568
8            Hours missing from resampled index      0
9               Hours with no valid consumption      1
10            Hours with no valid CO2 intensity      1

Duplicated timestamps — source rows (inspect before training):


                       time_utc  consommation  taux_co2  minute_utc
26490 2024-03-31 01:00:00+00:00       46662.0      20.0           0
26491 2024-03-31 01:00:00+00:00       46662.0      20.0           0
26489 2024-03-31 01:15:00+00:00           NaN       NaN          15
26488 2024-03-31 01:15:00+00:00           NaN       NaN          15
26486 2024-03-31 01:30:00+00:00       47346.0      20.0          30
26487 2024-03-31 01:30:00+00:00       47346.0      20.0          30
26485 2024-03-31 01:45:00+00:00           NaN       NaN          45
26484 2024-03-31 01:45:00+00:00           NaN       NaN          45

Missing 15-minute UTC timestamp slots (up to 12): [Timestamp('2024-10-27 00:00:00+0000', tz='UTC'), Timestamp('2024-10-27 00:15:00+0000', tz='UTC'), Timestamp('2024-10-27 00:30:00+0000', tz='UTC'), Timestamp('2024-10-27 00:45:00+0000', tz='UTC')]
Hours lacking valid targets (first 12) — consumption: [Timestamp('2024-10-27 00:00:00+0000', tz='UTC')] ; carbon: [Timestamp('2024-10-27 00:00:00+0000', tz='UTC')]
Note: source-null 15-minute rows can be expected when realized data is half-hourly.
ACTION: Resolve anomalies or document the exclusion policy before model training.


## 7. Stronger exploratory plot: seasonal relationships

The raw 2024 Pearson r = -0.575 for temperature vs national demand **mixes seasonal, weekday and diurnal effects**. We show the relationship by meteorological season, using **daily means** to make it easier to read, plus monthly correlations to check whether a single all-year correlation is hiding differences.

This remains descriptive, **not causal**; the 8-city temperature mean is unweighted.

In [28]:
if 'joined' in globals() and joined is not None and not joined.empty:
    s = joined[['mean_city_temp_C','consumption_MW','mean_city_wind_100m_kmh','co2_production_g_per_kWh']].copy()
    s['month'] = s.index.month
    def season(m):
        if m in [12,1,2]: return 'Winter'
        if m in [3,4,5]: return 'Spring'
        if m in [6,7,8]: return 'Summer'
        return 'Autumn'
    daily=s.resample('D').mean(numeric_only=True)
    daily['season']=[season(m) for m in daily.index.month]
    daily[['mean_city_temp_C','consumption_MW','season']].to_csv(OUT/'temperature_consumption_daily_2024.csv')
    fig,ax=plt.subplots(figsize=(9,5.5))
    for name in ['Winter','Spring','Summer','Autumn']:
        sub=daily[daily['season'].eq(name)].dropna(subset=['mean_city_temp_C','consumption_MW'])
        ax.scatter(sub['mean_city_temp_C'],sub['consumption_MW'],s=17,alpha=.75,label=f'{name} (n={len(sub)})')
    ax.set(title='France 2024 — daily temperature vs demand by season',xlabel='Unweighted average temperature, 8 cities (°C)',ylabel='French electricity consumption (MW)')
    ax.legend();ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(OUT/'fig_temp_consumption_by_season_2024.png',dpi=175);plt.show();plt.close(fig)
    monthly=[]
    for month,chunk in s.groupby('month'):
        pair=chunk[['mean_city_temp_C','consumption_MW']].dropna()
        monthly.append({'month':int(month),'hours':int(len(pair)),'pearson_r':float(pair.iloc[:,0].corr(pair.iloc[:,1])) if len(pair)>10 else np.nan})
    monthly=pd.DataFrame(monthly)
    monthly.to_csv(OUT/'monthly_temperature_consumption_correlations.csv',index=False)
    display(monthly.round(3))
else:
    print('Seasonal analysis skipped: a joined real-data dataframe is required.')

<Figure size 900x550 with 1 Axes>

    month  hours  pearson_r
0       1    744     -0.604
1       2    696     -0.007
2       3    744     -0.223
3       4    720     -0.228
4       5    744      0.202
5       6    720      0.641
6       7    744      0.720
7       8    744      0.714
8       9    720      0.367
9      10    743      0.330
10     11    720     -0.322
11     12    744     -0.181

## 8. Leakage demonstration: archived J-1 forecast versus verified ERA5 weather

The first seven days of October 2024 were retrieved from Open-Meteo's Previous Runs API in section 4. **Compare those fixed-lead weather predictions to subsequent ERA5 observed/reanalysis values** for Paris and calculate MAE. A 24-hour-lead forecast is *not* the same as reconstructing all forecasts available at a single daily decision time; a full operational J+1 protocol must define issue time and archives accordingly.

This is a real pilot, **not evidence for 2019–2023 historical forecast coverage**.

In [29]:
if 'piloth' in globals() and 'wx_raw' in globals() and wx_raw is not None:
    past=piloth.copy()
    past['time_utc']=pd.to_datetime(past['time'],utc=True,errors='coerce')
    obs=wx_raw.loc[wx_raw['city'].eq('Paris'),['time_utc','temperature_2m','wind_speed_100m']]
    cmp=past.merge(obs,on='time_utc',how='inner',validate='one_to_one')
    stats=[]
    fig,axs=plt.subplots(2,1,figsize=(10,6.8),sharex=True)
    for ax,(a,b,units,label) in zip(axs,[
        ('temperature_2m_previous_day1','temperature_2m','°C','Temperature at 2 m'),
        ('wind_speed_100m_previous_day1','wind_speed_100m','km/h','Wind speed at 100 m')
    ]):
        valid=cmp[['time_utc',a,b]].dropna()
        if len(valid) == 0: continue
        err=valid[a]-valid[b]
        stats.append({'weather_variable':label,'valid_hourly_pairs':len(valid),
                      'MAE':float(err.abs().mean()),'RMSE':float(np.sqrt((err**2).mean())),
                      'units':units})
        ax.plot(valid['time_utc'],valid[b],label='Observed ERA5',linewidth=1.7)
        ax.plot(valid['time_utc'],valid[a],label='Archived forecast (previous_day1)',linewidth=1.3,alpha=.85)
        ax.set_ylabel(f'{label} ({units})');ax.grid(alpha=.15);ax.legend(loc='upper right',fontsize=8)
    fig.suptitle('Paris, 1–7 October 2024 — 24 h ahead forecast vs reanalysis (pilot)')
    fig.autofmt_xdate();fig.tight_layout();fig.savefig(OUT/'fig_previous_day1_vs_era5_pilot.png',dpi=175)
    plt.show();plt.close(fig)
    stats=pd.DataFrame(stats)
    stats.to_csv(OUT/'weather_forecast_error_pilot.csv',index=False)
    print('Matched forecast–reanalysis pairs:',len(cmp));display(stats.round(3))
else:
    print('Pilot comparison unavailable; no historical forecast errors fabricated.')

<Figure size 1000x680 with 2 Axes>

Matched forecast–reanalysis pairs: 168


      weather_variable  valid_hourly_pairs    MAE   RMSE units
0   Temperature at 2 m                 168  0.950  1.193    °C
1  Wind speed at 100 m                 168  4.138  5.197  km/h

## 9. Mechanism check: wind → actual generation → grid CO₂

A near-zero correlation between wind speed and national CO₂ intensity does **not** show wind power is ineffective. Test the more specific link between unweighted eight-city wind speed and actual French wind generation, then between wind generation and RTE's **generation-based average** CO₂ intensity. Show coefficients and sample sizes explicitly. These are observational relationships and may be confounded by load, imports, seasonality and the rest of the production mix.


In [30]:
try:
    details_query = {
        'select':'date_heure,eolien,gaz,solaire,nucleaire,hydraulique',
        'where':"date_heure >= '2024-01-01T00:00:00Z' AND date_heure < '2025-01-01T00:00:00Z'",
        'delimiter':';'
    }
    rd=S.get(RTE_EXPORT,params=details_query,timeout=180)
    rd.raise_for_status()
    prod=pd.read_csv(io.BytesIO(rd.content),sep=None,engine='python',dtype=str,encoding='utf-8-sig')
    prod.columns=[x.strip().replace('\ufeff','') for x in prod.columns]
    expected=['date_heure','eolien','gaz','solaire','nucleaire','hydraulique']
    if not set(expected).issubset(prod.columns):
        raise ValueError('Expected production fields unavailable: ' + str(prod.columns.tolist()[:20]))
    prod['time_utc']=pd.to_datetime(prod.pop('date_heure'),utc=True,errors='coerce')
    for col in expected[1:]:
        prod[col]=pd.to_numeric(prod[col].str.replace(',','.',regex=False),errors='coerce')
    prod=prod.dropna(subset=['time_utc']).set_index('time_utc').sort_index()
    prod_hourly=prod[expected[1:]].resample('h').mean()
    check_data=joined.join(prod_hourly,how='inner')
    check_data.to_csv(OUT/'joined_production_mix_2024_hourly.csv')
    wind_diagnostics=[]
    fig,axs=plt.subplots(1,2,figsize=(12,4.8))
    for ax,(x,y,xlabel,ylabel) in zip(axs,[
        ('mean_city_wind_100m_kmh','eolien','8-city unweighted wind at 100 m (km/h)','RTE actual wind generation (MW)'),
        ('eolien','co2_production_g_per_kWh','RTE actual wind generation (MW)','Production-based CO2 intensity (g/kWh)')
    ]):
        pairs=check_data[[x,y]].dropna()
        r=float(pairs[x].corr(pairs[y]))
        wind_diagnostics.append({'relationship':x+' vs '+y,'n':int(len(pairs)),'pearson_r':r})
        ax.scatter(pairs[x],pairs[y],s=3,alpha=.18,rasterized=True)
        ax.set(xlabel=xlabel,ylabel=ylabel,title=f'2024 exploratory r={r:.3f}, n={len(pairs)}')
        ax.grid(alpha=.15)
    fig.suptitle('Weather proxy → actual wind production → grid CO2 intensity (descriptive only)')
    fig.tight_layout();fig.savefig(OUT/'fig_wind_proxy_vs_generation_and_carbon_2024.png',dpi=175)
    plt.show();plt.close(fig)
    wind_diagnostics=pd.DataFrame(wind_diagnostics)
    wind_diagnostics.to_csv(OUT/'wind_generation_relationships_2024.csv',index=False)
    display(wind_diagnostics.round(3))
except Exception as e:
    print('Optional production-mix diagnostic not completed:',type(e).__name__,str(e)[:350])
    print('Do not claim this plot exists until the RTE API request succeeds.')


<Figure size 1200x480 with 2 Axes>

                         relationship     n  pearson_r
0   mean_city_wind_100m_kmh vs eolien  8783      0.861
1  eolien vs co2_production_g_per_kWh  8783      0.022

## 10. Evaluation 1 — what has actually been established

The **executed audited v3** supports: eight-city ERA5 2024 samples, RTE 2024 data, two exploratory plots, seasonal relationships, 168 matched fixed-lead forecast/reanalysis pairs for Paris (1–7 October), wind-generation diagnostics, and the corrected RTE cadence/timestamp quality audit.

**Critical decisions for Ramy and supervisors:**

1. **Forecast target:** RTE production-based `taux_co2` (g/kWh) or electricity demand (MW)? Forecast one as primary and keep the other secondary.
2. **Issue time + horizon:** specify the UTC time when a J+1 forecast would be generated. Do not use observations that became available later.
3. **Temporal split:** 2019–2023 / 2024 / **2025 locked** is suitable for lag/calendar baselines. Historical weather-forecast coverage may require a separate approved operational split; an archived 2024 pilot is not evidence of 2019 coverage.
4. **Geography:** eight city points with equal weights are exploratory proxies. Later use sourced population / wind generation-capacity weights.
5. **Emissions claims:** differences in average French generation-based intensity can estimate *illustrative* benefits; they are not evidence of marginal avoided emissions caused by shifting demand.
6. **RTE time quality:** document duplicated UTC timestamps and missing/blank intervals before building models.

**Presentation-ready message:** "In 2024, temperature's association with electricity demand changes substantially by season. Weather wind speed strongly tracks actual wind production, but alone explains little about national average production-based CO₂ intensity. A pilot confirms the availability and imperfect accuracy of archived fixed-lead weather forecasts; the future operational protocol will fix the issue time and avoid weather look-ahead leakage."


## 11. Automatically generated Phase 0 evidence card

The following report is generated **only from actual in-memory results**. It creates a short text summary for sharing with Ramy and records the scientific limitations. If a source was unavailable, the report marks the corresponding result unavailable.


In [31]:
import json
from datetime import datetime, timezone

summary={'report':'GreenGrid AI Aziz Phase 0 (2024 data; no 2025 access)',
         'generated_utc':datetime.now(timezone.utc).isoformat(),
         'methodology':'Descriptive EDA only; no causal inference or trained day-ahead predictor'}
if 'joined' in globals() and joined is not None and not joined.empty:
    summary['joined_hours']=int(len(joined))
    summary['temp_consumption_r']=round(float(joined['mean_city_temp_C'].corr(joined['consumption_MW'])),3)
    summary['wind_proxy_co2_r']=round(float(joined['mean_city_wind_100m_kmh'].corr(joined['co2_production_g_per_kWh'])),3)
if 'monthly' in globals() and isinstance(monthly,pd.DataFrame):
    summary['monthly_temp_demand_r']={str(int(row.month)):round(float(row.pearson_r),3) for row in monthly.itertuples()}
if 'stats' in globals() and isinstance(stats,pd.DataFrame):
    summary['forecast_pilot']=[{'variable':str(row.weather_variable),'hours':int(row.valid_hourly_pairs),
                                'mae':round(float(row.MAE),3),'rmse':round(float(row.RMSE),3),
                                'units':str(row.units)} for row in stats.itertuples()]
if 'qa' in globals() and isinstance(qa,pd.DataFrame):
    summary['rte_quality']={str(row.check):int(row.value) for row in qa.itertuples()}
if 'wind_diagnostics' in globals() and isinstance(wind_diagnostics,pd.DataFrame):
    summary['wind_diagnostics']=[{'relationship':str(row.relationship),'n':int(row.n),
                                 'r':round(float(row.pearson_r),3)} for row in wind_diagnostics.itertuples()]
(OUT/'evaluation1_evidence_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
lines=['# Aziz — GreenGrid AI, Phase 0: verified evidence','',
       'This document was generated from executed Kaggle cells, not estimated values.','',
       '## Numerical results', '',
       '```json',json.dumps({k:v for k,v in summary.items() if k not in ['report','generated_utc','methodology']},ensure_ascii=False,indent=2),'```',
       '', '## Important caveats',
       '- Eight city weather points are unweighted and are not a representative French weather aggregation.',
       '- ERA5 at the target hour is reanalysis, not information available when a day-ahead forecast is issued.',
       '- Previous-day1 fixed-lead weather forecasts are not a reconstructed single daily issue-time snapshot.',
       '- RTE average generation-based CO2 is not marginal emissions or necessarily consumption-based intensity.',
       '- One pilot week is not enough to establish general forecast skill.',
       '- Correlations are descriptive and do not demonstrate causality.',
       '- Inspect and resolve abnormal or duplicated UTC timestamps and any missing targets before training.',
       '- 2025 is the proposed locked test and has not been accessed.']
(OUT/'EVALUATION_1_EVIDENCE.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('Evidence card saved to:',OUT/'EVALUATION_1_EVIDENCE.md')
print(json.dumps({k:v for k,v in summary.items() if k != 'monthly_temp_demand_r'},ensure_ascii=False,indent=2))


Evidence card saved to: /kaggle/working/greengrid_phase0/EVALUATION_1_EVIDENCE.md
{
  "report": "GreenGrid AI Aziz Phase 0 (2024 data; no 2025 access)",
  "generated_utc": "2026-11-01T14:22:27.102453+00:00",
  "methodology": "Descriptive EDA only; no causal inference or trained day-ahead predictor",
  "joined_hours": 8784,
  "temp_consumption_r": -0.575,
  "wind_proxy_co2_r": -0.007,
  "forecast_pilot": [
    {
      "variable": "Temperature at 2 m",
      "hours": 168,
      "mae": 0.95,
      "rmse": 1.193,
      "units": "°C"
    },
    {
      "variable": "Wind speed at 100 m",
      "hours": 168,
      "mae": 4.138,
      "rmse": 5.197,
      "units": "km/h"
    }
  ],
  "rte_quality": {
    "Export rows": 35136,
    "Unique UTC timestamps": 35132,
    "Duplicate UTC timestamps (extra rows)": 4,
    "Expected UTC 15-minute slots": 35136,
    "Missing UTC 15-minute slots": 4,
    "Unexpected UTC timestamps off expected grid": 0,
    "Null consumption export rows": 17568,
    "Null 